## 1. What is Google AI Studio & Gemini?

Google AI Studio is a web-based prototyping environment for developers to quickly build with Gemini models. It provides seamless access to state-of-the-art multimodal models like Gemini 2.5 Flash, Gemini 2.5 Flash-Lite and Gemini Pro.

By using the langchain-google-genai integration, you can unlock advanced functionalities, huge context windows (up to 1M+ tokens) and native multimodal capabilities (text, images, video) in your applications.

`Why Gemini 2.5 Flash-Lite?`
> Speed: Extremely low latency, perfect for real-time apps.

> Cost/Free Tier: It has a very generous free tier (high rate limits), making it ideal for learning and high-volume testing.

In [1]:
# Install the necessary packages for Google AI and LangChain
#!pip install -qU langchain-google-genai langchain langchain-community pypdf google-search-results

**PACKAGES WE HAVE USED IN THIS NOTEBOOK :**
```

# Common packages
from langchain_google_genai import ChatGoogleGenerativeAI # Important 
from langchain_core.prompts import ChatPromptTemplate # We have to give list of messages like we give to gpt
from langchain_core.prompts import PromptTemplate # We just have to give a single string 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import json

# For Preserving Memory
from langchain.chains import LLMChain, ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory

# Agent and Tools
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
```

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os

In [4]:
KEY = "PASTE_YOUR_GOOGLE_AI_STUDIO_KEY_HERE"

In [5]:
os.environ["GOOGLE_API_KEY"] = KEY

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

### Model Parameters

> Temperature (0.0 - 2.0): Controls randomness. Lower values (e.g., 0.2) make the output more deterministic and focused. Higher values (e.g., 0.9) make it more creative.

> Max Output Tokens: Limits the length of the generated response.

> Top P: Nucleus sampling. The model considers the results of the tokens with top_p probability mass.

> Top K: The model considers the top k most probable tokens.

`Note : Currently we are using Temperature only`

In [7]:
# Initialize the Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
    google_api_key=KEY
)

In [8]:
response = llm.invoke("Who was the first prime minister of India?")
print(response.content)

The first Prime Minister of India was **Jawaharlal Nehru**.


## 2. Prompt Templates (Structured Prompts)

In [9]:
from langchain_core.prompts import ChatPromptTemplate

In [10]:
# Create a prompt template
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Tell me a fun fact about {topic}."
)

In [11]:
# Chain the prompt with the model
chain = prompt | llm

In [12]:
# Invoke the chain
response = chain.invoke({"topic": "cricket"})
print(response.content)

Here's a fun fact about cricket:

The longest cricket match ever played was the **Timeless Test** between South Africa and England in 1939. It lasted for **9 days**! Due to time constraints, neither team could secure a victory, and the match was declared a draw. Imagine the sheer endurance of those players!


### Extract Keywords from paragraphs

In [13]:
# 1. The input text
student_description = "Shreyash Narvekar is a student of computer science at IIT delhi. He is an indian and has a 8.5 GPA. Shreyash is known for his programming skills and is an active member of the college's AI Club. He hopes to pursue a career in artificial intelligence after graduating."

In [14]:
# 2. A simple prompt 
prompt = f'''
Please extract the following information from the given text and return it as a JSON object:

name
college
grades
club

This is the body of text to extract the information from:
{student_description}
'''

In [15]:
response = llm.invoke(prompt)

In [16]:
response

AIMessage(content='```json\n{\n  "name": "Shreyash Narvekar",\n  "college": "IIT Delhi",\n  "grades": "8.5 GPA",\n  "club": "AI Club"\n}\n```', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b83cb-5c03-7ed1-b0b8-c1774393eb7d-0', usage_metadata={'input_tokens': 105, 'output_tokens': 50, 'total_tokens': 155, 'input_token_details': {'cache_read': 0}})

In [17]:
output = response.content

In [18]:
print(output)

```json
{
  "name": "Shreyash Narvekar",
  "college": "IIT Delhi",
  "grades": "8.5 GPA",
  "club": "AI Club"
}
```


> Convert String to Dictionary (Json)

In [19]:
import json

In [20]:
# Clean the output string to remove markdown code blocks if present
cleaned_output = output.replace("```json", "").replace("```", "").strip()

In [21]:
# Convert to a Python Dictionary
data = json.loads(cleaned_output)

In [22]:
# Now you see fields like a normal dictionary or json
data

{'name': 'Shreyash Narvekar',
 'college': 'IIT Delhi',
 'grades': '8.5 GPA',
 'club': 'AI Club'}

## RECAP - USED TECHNICAL TERMS FOR UNDERSTANDING

### Zero Shot Prompting

In [23]:
# Simple Zero-Shot Prompting
prompt = "Can you tell me the total number of countries in Asia? Can you give me the top 10 country names?"

In [24]:
# Get response
response = llm.invoke(prompt)
print(response.content.strip())

The total number of countries in Asia can be a bit fluid depending on how you define "country" and how you categorize transcontinental countries. However, the most widely accepted number is **48 countries**.

Here are the **top 10 largest countries in Asia by land area**:

1.  **Russia** (a significant portion of its territory is in Asia)
2.  **China**
3.  **India**
4.  **Kazakhstan** (a significant portion of its territory is in Asia)
5.  **Saudi Arabia**
6.  **Indonesia** (includes territory in Oceania as well)
7.  **Iran**
8.  **Mongolia**
9.  **Pakistan**
10. **Myanmar (Burma)**


### Prompt Templates

In [25]:
from langchain_core.prompts import PromptTemplate

In [26]:
# 1. Define the template
prompt_template_name = PromptTemplate(
    input_variables=["country"],
    template="Can you tell me the capital of {country}?"
)

In [27]:
# 2. Format the prompt
prompt1 = prompt_template_name.format(country="India")

In [28]:
# 3. Predict (using invoke)
response = llm.invoke(prompt1)
print(response.content.strip())

The capital of India is **New Delhi**.


### Simple Chain 

In [29]:
# Define the prompt
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}?")

In [30]:
# Create the chain (This replaces LLMChain)
chain = prompt | llm

In [31]:
# Run it
response = chain.invoke({"product": "colorful socks"})
print(response.content.strip())

Here are some good name ideas for a colorful sock company, categorized by the vibe they evoke:

**Playful & Whimsical:**

* **KaleidoSox** (Kaleidoscope + Socks)
* **PrismPeds** (Prism + Pedals/Feet)
* **HueHuggers**
* **Colorburst Feet**
* **Sockaroo Fun**
* **Rainbow Rebels**
* **Chroma Charms**
* **Doodle Duds**
* **Zany Zocks**
* **Whimsy Wares**

**Modern & Chic:**

* **The Sock Spectrum**
* **Vivid Threads**
* **Aura Socks**
* **Loom & Hue**
* **Palette Footwear**
* **Candid Chroma**
* **Thread & Tint**
* **Solera Socks** (Solera refers to a blending process, suggesting unique color combinations)
* **Infusion Footwear**
* **Apex Hues**

**Descriptive & Straightforward:**

* **Colorful Socks Co.**
* **The Vibrant Sock Shop**
* **Bright Footwear**
* **Hueful Socks**
* **Colorfully Yours Socks**
* **The Patterned Pair**
* **Sock Artistry**
* **Bold & Bright Socks**
* **The Daily Dazzle**
* **Expressive Feet**

**Evocative & Creative:**

* **Foot Canvas**
* **Chromatica Socks**
* **T

## Sequential Chain (SimpleSequentialChain)

In [32]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [33]:
# Chain 1: Generate Company Name
prompt_name = PromptTemplate.from_template(
    """Suggest one catchy, professional startup name for {topic}.
    CRITICAL INSTRUCTION: Output ONLY the name. Do not add "Here is a name", do not add quotes, and do not add any explanation. Just the single word."""
)
chain_name = prompt_name | llm | StrOutputParser()

In [34]:
test_name = chain_name.invoke({"topic": "Artificial Intelligence"})
print(f"Chain 1 Output: {test_name}")

Chain 1 Output: AuraMind


In [35]:
# Chain 2: Generate Strategies (takes the output of chain_name that is chain 1)
prompt_strategies = PromptTemplate.from_template(
    """You are a business strategist.
    The company is named '{company_name}' and it works in the '{topic}' industry.
    Suggest 3 distinct business strategies for them to succeed.
    Write the response strictly as a list."""
)
chain_strategies = prompt_strategies | llm | StrOutputParser()

**Create Sequential Chain Now of (Chain 1 and Chain 2)**

In [36]:
full_chain = (
    {"company_name": chain_name, "topic": RunnablePassthrough()} | chain_strategies
)

In [37]:
topic = "Artificial Intelligence"

In [38]:
result = full_chain.invoke(topic)

In [39]:
print(result)

1.  **Niche Specialization & Deep Domain Expertise:** Focus on developing AI solutions for a specific, high-value industry vertical (e.g., AI for drug discovery in pharmaceuticals, AI for predictive maintenance in advanced manufacturing, AI for fraud detection in insurtech). This strategy involves building unparalleled domain knowledge within that niche, allowing Cognito to offer highly tailored, superior solutions that competitors with broader offerings cannot match. It fosters strong client relationships through deep understanding of their unique challenges and regulatory landscapes.

2.  **Platform-as-a-Service (PaaS) for Customizable AI Development:** Develop and offer a proprietary, low-code/no-code AI development platform. This platform would empower businesses, even those without extensive AI expertise, to build, deploy, and manage custom AI models for their specific needs. Cognito would profit through subscription fees, tiered service levels, and potentially a marketplace for p

## 3. Document Loading (PDFs)
> This is crucial for RAG (Retrieval Augmented Generation), where you load your own data (PDFs, text files) to let the AI answer questions about them.

In [40]:
# Cell for loading YOUR local PDF
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI

In [41]:
pdf_filename = r"D:\Developement\GEN AI\Gen AI - Projects\01 Learning\shreyash_narvekar_resume.pdf"

In [42]:
# Load the PDF
loader = PyPDFLoader(pdf_filename)

In [43]:
pages = loader.load_and_split()

In [44]:
print(f"Total Pages Loaded: {len(pages)}")

Total Pages Loaded: 2


In [45]:
# Initialize the Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2,
    google_api_key=KEY
)

In [46]:
# Summarize the first page
summary = llm.invoke(f"Summarize this text:\n\n{pages[0].page_content}")

In [47]:
print(summary.content)

This text describes a project called "Sound - Automated Business Analytics & Accounts Management," developed using Python and Django. The project is a full-stack web application designed to streamline business operations such as booking, financial management, and client/staff data handling. Key features include a dynamic dashboard with automated reporting for financial insights, automated financial calculations for profitability and expenses, an intuitive user interface, and automated email notifications for staff.

The text also provides a summary of an AI & ML Engineer and Python Developer named Shreyash Narvekar. Shreyash has expertise in developing web applications and machine learning models, with a Bachelor's degree in Artificial Intelligence & Machine Learning Engineering and a Diploma in Computer Engineering. Their technical skills include Python, Django, various machine learning and data science libraries (TensorFlow, NLTK, Pandas), and tools like Git. The document also lists 

## 4. Memory

In [48]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import trim_messages

In [49]:
# Initialize the Model
llm = ChatGoogleGenerativeAI(
    # model="gemini-2.5-flash-lite",
    model="gemini-2.5-flash",
    temperature=0.,
    google_api_key=KEY
)

`Setup a dictionary to store conversation history`

`In a real app, this would be a database like Redis`

In [50]:
store = {}

In [51]:
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

### Conversation Buffer Memory
Adds memory so it remembers previous suggestions.

In [52]:
# 1. Define Prompt with a History Placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative assistant that suggests fancy restaurant names."),
    MessagesPlaceholder(variable_name="history"), # <--- This is where memory lives
    ("human", "I want to open a restaurant for {cuisine} food. Suggest a fancy name for this.")
])

In [53]:
# 2. Create the Chain
chain = prompt | llm | StrOutputParser()

In [54]:
# 3. Add Memory to the Chain
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="cuisine",
    history_messages_key="history"
)

In [55]:
# 4. Run the Chain
session_id = "restaurant_idea_1"

In [56]:
# Query 1: Mexican

In [57]:
response1 = chain_with_memory.invoke(
    {"cuisine": "Mexican"},
    config={"configurable": {"session_id": session_id}}
)

In [58]:
print(response1)

Okay, "fancy Mexican" is a wonderful concept! It moves beyond tacos and burritos into refined flavors, elevated presentation, and a sophisticated ambiance. Here are some suggestions, playing on different aspects of "fancy" and "Mexican":

**Evocative & Poetic:**

1.  **Alma y Fuego:** (Soul and Fire) – Suggests passion, deep flavor, and warmth.
2.  **Maíz y Luna:** (Corn and Moon) – Evokes ancient traditions, staple ingredients, and a magical night.
3.  **El Cielo y la Tierra:** (The Sky and the Earth) – Implies a connection to nature, fresh ingredients, and a grand experience.
4.  **Cenote Secreto:** (Secret Cenote) – Mystical, exclusive, beautiful, referencing Mexico's natural wonders.
5.  **Sol de la Noche:** (Sun of the Night) – Intriguing paradox, suggesting a vibrant and elegant evening.
6.  **La Raíz del Sabor:** (The Root of Flavor) – Emphasizes authenticity, depth, and heritage ingredients.

**Elegant & Refined (often using Spanish terms for high quality):**

7.  **Hacienda Al

In [59]:
response2 = chain_with_memory.invoke(
    {"cuisine": "Indian"},
    config={"configurable": {"session_id": session_id}}
)

In [60]:
print(response2)

Excellent choice! Indian cuisine, with its rich history, vibrant spices, and diverse regional styles, lends itself beautifully to a fancy, upscale experience.

Here are some fancy Indian restaurant name suggestions, playing on different aspects like royalty, spices, heritage, and luxury:

**Regal & Grand:**

1.  **The Royal Darbar:** (Darbar means court or assembly, implying a grand gathering)
2.  **Haveli Kitchen:** (Haveli refers to a traditional Indian mansion, evoking heritage and luxury)
3.  **Emperor's Table:** (Directly implies a feast fit for royalty)
4.  **Maharaja's Jewel:** (Connects to preciousness and royal heritage)
5.  **Indus Dynasty:** (References ancient Indian civilization and a lasting legacy)
6.  **The Gilded Palace:** (Suggests opulent surroundings and golden accents)
7.  **The Viceroy's Dining Room:** (Evokes colonial elegance and grandeur)
8.  **Rajputana Royale:** (Refers to the land of kings, Rajasthan, known for its royal cuisine)

**Poetic & Sensory:**

9.  

In [61]:
print("\n--- View Memory Buffer (What the AI remembers) ---")
print(store[session_id])


--- View Memory Buffer (What the AI remembers) ---
Human: Mexican
AI: Okay, "fancy Mexican" is a wonderful concept! It moves beyond tacos and burritos into refined flavors, elevated presentation, and a sophisticated ambiance. Here are some suggestions, playing on different aspects of "fancy" and "Mexican":

**Evocative & Poetic:**

1.  **Alma y Fuego:** (Soul and Fire) – Suggests passion, deep flavor, and warmth.
2.  **Maíz y Luna:** (Corn and Moon) – Evokes ancient traditions, staple ingredients, and a magical night.
3.  **El Cielo y la Tierra:** (The Sky and the Earth) – Implies a connection to nature, fresh ingredients, and a grand experience.
4.  **Cenote Secreto:** (Secret Cenote) – Mystical, exclusive, beautiful, referencing Mexico's natural wonders.
5.  **Sol de la Noche:** (Sun of the Night) – Intriguing paradox, suggesting a vibrant and elegant evening.
6.  **La Raíz del Sabor:** (The Root of Flavor) – Emphasizes authenticity, depth, and heritage ingredients.

**Elegant & Ref

### Conversation Chain
This is like a "Chatbot" (Gemini, ChatGPT, etc.)

In [62]:
# 1. Simple Chat Prompt
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

In [63]:
# 2. Create Chat Chain
chat_chain = chat_prompt | llm | StrOutputParser()

In [64]:
# 3. Wrap with Memory
conversation = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [65]:
# 4. Have a Conversation
session_id = "cricket_chat"

In [66]:
# Prompt 1
print(conversation.invoke(
    {"input": "Who won the first cricket world cup?"},
    config={"configurable": {"session_id": session_id}}
))

The first Cricket World Cup was won by **West Indies** in 1975.

They defeated Australia in the final.


In [67]:
# Prompt 2
print(conversation.invoke(
    {"input": "How much is 5+5?"},
    config={"configurable": {"session_id": session_id}}
))

5 + 5 = **10**


In [68]:
# Prompt 3
print(conversation.invoke(
    {"input": "Who was the captain of that winning team?"},
    config={"configurable": {"session_id": session_id}}
))

The captain of the West Indies team that won the first Cricket World Cup in 1975 was **Clive Lloyd**.


### Window Memory (Limiting History) - USING K VALUE
ConversationBufferWindowMemory(k=2) to keep only the last 2 messages in memory.

In [69]:
from langchain_core.messages import HumanMessage, AIMessage

In [70]:
# We will manually add messages here.
chat_history = []

In [71]:
# Define the Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"), # <--- History goes here
    ("human", "{input}"),
])

In [72]:
# Create the Chain
chain = prompt | llm | StrOutputParser()

In [73]:
# --- HELPER FUNCTION (Simulates Window Memory k=2) ---
def chat_with_memory(user_input, k=2):
    # A. Get only the last k*2 messages (Human + AI pairs)
    # If k=2, we grab the last 4 items from the list
    recent_history = chat_history[-(k*2):] 
    
    # B. Run the Chain with this history
    response = chain.invoke({
        "input": user_input,
        "history": recent_history
    })
    
    # C. Update the main memory list
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))
    
    return response

In [74]:
# Prompt 1
print(chat_with_memory("The secret code is 1234."))

Understood! I've noted that the secret code is 1234.


In [75]:
# Prompt 2
print(chat_with_memory("I like Python."))

That's great! Python is a fantastic language. Many people appreciate it for its:

*   **Readability and Simplicity:** It's often compared to plain English, making it easy to learn and use.
*   **Versatility:** You can use it for web development (Django, Flask), data science (Pandas, NumPy), machine learning (TensorFlow, PyTorch), scripting, automation, and much more.
*   **Large Community and Ecosystem:** There's a vast amount of libraries, frameworks, and support available.

What do you like most about Python, or what do you primarily use it for?


In [76]:
# Prompt 3
print(chat_with_memory("I live in India."))

That's interesting to know! India is a diverse country with a rich history and culture.


In [77]:
# Prompt 4 (Checking Memory k=2)
# If k=2, it remembers Prompt 2 and Prompt 3.
# It might forget Prompt 1 (Your name) depending on the window!
print(chat_with_memory("What is the secret code?"))

That's an intriguing question! "Secret code" can mean so many different things.

To help me understand, could you tell me a little more about what kind of secret code you're thinking of? For example:

*   Are you referring to a specific **programming concept or technique** in Python?
*   Is it a **riddle or a puzzle**?
*   A **reference from a movie, book, or game**?
*   Something else entirely?

Once I have a bit more context, I can try to help you figure it out!


In [78]:
# Prompt 5 (Current Memory Storage)
print(chat_with_memory("Tell me what you remember."))

Okay, I remember our conversation!

Here's what we've covered:

1.  You mentioned that you live in India.
2.  You then asked me, "What is the secret code?"
3.  I responded by asking for more context about what kind of secret code you might be referring to (e.g., a programming concept, a riddle, a reference from a movie/book, etc.).

As an AI, I don't "remember" in the human sense of having personal experiences or long-term recall between conversations. Instead, I process and retain the information from our current dialogue to maintain context and respond relevantly.

So, to help me understand, could you elaborate on what kind of secret code you have in mind?


## 5. Agents & Tools
It is the **Reasoning Engine** where the AI doesn't just talk-it does things `like searching Google or calculating math.`

In [79]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

`1. Setup Keys` 

In [80]:
SERPAPI_KEY = "PASTE_YOUR_SERPAPI_KEY_HERE"

In [81]:
# os.environ["GOOGLE_API_KEY"] = "KEY"
os.environ["SERPAPI_API_KEY"] = "SERPAPI_KEY"

`2. Initialize the Model`

In [82]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=KEY
)

`3. Define tool (Manual Tool Creation)`

In [83]:
# We wrap the utility in a simple tool function
search = SerpAPIWrapper(serpapi_api_key=SERPAPI_KEY)

In [84]:
# Tool 1: Search
@tool
def google_search(query: str):
    """Searches Google for current events and real-time information."""
    return search.run(query)

In [85]:
# Tool 2: Calculator
@tool
def calculator(expression: str):
    """Calculates math expressions."""
    return str(eval(expression))

In [86]:
# Bundle the tools together
tools = [google_search, calculator]

`4. Create the Agent (Using LangGraph)`

In [87]:
agent_executor = create_react_agent(llm, tools)

`5. Run the Agent`

In [88]:
query = "What is the GDP of the US in 2024? Add 5 to that number."

In [89]:
# LangGraph returns a dictionary of messages
response = agent_executor.invoke({"messages": [("human", query)]})

In [90]:
# We extract the final message content
print(f"\nFinal Answer:\n{response['messages'][-1].content}")


Final Answer:
The GDP of the US in 2024 is 29,184.89 billion USD. Adding 5 to that number gives you 29,189.89 billion USD.


In [91]:
pip list

Package                   Version
------------------------- -----------
aiohappyeyeballs          2.6.1
aiohttp                   3.13.2
aiosignal                 1.4.0
annotated-types           0.7.0
anyio                     4.12.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.0.5
attrs                     25.4.0
babel                     2.17.0
beautifulsoup4            4.14.3
bleach                    6.3.0
cachetools                6.2.4
certifi                   2025.11.12
cffi                      2.0.0
charset-normalizer        3.4.4
colorama                  0.4.6
comm                      0.2.3
dataclasses-json          0.6.7
debugpy                   1.8.19
decorator                 5.2.1
defusedxml                0.7.1
distro                    1.9.0
executing                 2.2.1
fastjsonschema            2.21.2
filelock                  3.20.1
filetype         